<a href="https://colab.research.google.com/github/AmandineBaranger/Group_06_DS_Project/blob/main/notebooks/02_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetes 130-US Hospitals - Baseline Models and Evaluation

- Part 1 (sections 1 to 3): we load the prepared train and test sets and we define our validation strategy grouped by patient.
- Part 2 (sections 4 to 7): we build and evaluate three baseline models (Dummy Classifier, Logistic Regression, and Decision Tree) using ROC-AUC, PR-AUC, Recall, Precision, and F1-score.
- Part 3 (sections 8 to 9): we compare all baselines on the validation folds, check error patterns, and save our reference results for Week 3.

## 1. Imports and paths

In [1]:
# import os to handle file paths and folders
import os
# import numpy for numerical calculations
import numpy as np
# import pandas to load and manipulate tables
import pandas as pd
# import matplotlib to draw charts
import matplotlib.pyplot as plt
# import seaborn for clean visualisations
import seaborn as sns

# import scikit-learn tools for grouped cross-validation without data leakage
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
# import simple baseline classifiers
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
# import evaluation metrics adapted to imbalanced binary classification
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    RocCurveDisplay,
    PrecisionRecallDisplay
)

# use a clean style for all charts
sns.set_theme(style="whitegrid")
# show all columns when printing tables
pd.set_option("display.max_columns", 60)

# relative paths to the processed data and outputs folders
PROCESSED_DIR = os.path.join("..", "data", "processed")
# fallback URL on GitHub if running in Google Colab without local processed files
GITHUB_PROCESSED_URL = "https://raw.githubusercontent.com/AmandineBaranger/Group_06_DS_Project/main/data/processed"

# fixed random seed so all splits and models are reproducible
SEED = 42

What this code does: we import the required libraries, set a fixed random seed (`SEED = 42`), and define relative paths to load the preprocessed train and test datasets created in the first module. We also define the GitHub raw URL as a fallback so the notebook runs seamlessly both on local Jupyter setups and on Google Colab.

## 2. Loading the processed train and test sets

In [2]:
# helper function to load from local processed folder first, or from GitHub if on Colab
def load_processed_csv(filename):
    local_path = os.path.join(PROCESSED_DIR, filename)
    if os.path.exists(local_path):
        return pd.read_csv(local_path)
    return pd.read_csv(f"{GITHUB_PROCESSED_URL}/{filename}")

# load training and test features and targets produced by Part 2 of notebook 01
X_train = load_processed_csv("X_train.csv")
X_test = load_processed_csv("X_test.csv")
y_train = load_processed_csv("y_train.csv").squeeze()
y_test = load_processed_csv("y_test.csv").squeeze()

# load patient identifiers for the training set to prevent patient leakage during cross-validation
groups_train = load_processed_csv("groups_train.csv").squeeze()

# print dataset shapes and target proportions to verify consistency
print("X_train shape :", X_train.shape)
print("X_test shape  :", X_test.shape)
print("Train positive rate (<30 days) :", round(y_train.mean(), 4))
print("Test positive rate (<30 days)  :", round(y_test.mean(), 4))
print("Unique patients in train       :", groups_train.nunique())

HTTPError: HTTP Error 404: Not Found